# 清理旧数据

In [7]:
from langgraph_sdk import get_client

client = get_client(
    url="http://localhost",
    headers={"Authorization": "Bearer admin"},
) 
assistant_id = 'd675cd61-4be4-5882-ab8e-5ae49a2846bd'

In [8]:
# 删除所有线程
deleted_threads = 0
while threads := await client.threads.search(limit=100):
    for thread in threads:
        await client.threads.delete(thread["thread_id"])
        deleted_threads += 1
print(f"已删除 {deleted_threads} 个线程")

# 新建线程，方便观察结果
thread = await client.threads.create(
    metadata={
        "__name__":"删除旧的trace"
    }
)
thread_id = thread.get("thread_id")
thread_id

已删除 2 个线程


'01a0b35f-6f5d-7490-a5cb-20e900502197'

In [13]:
# 删除所有 Cron（先停止定时任务，避免清理线程时继续创建新线程）
crons = await client.crons.search(limit=100)
for cron in crons:
    await client.crons.delete(cron.get("cron_id"))

In [12]:
# 测试：每秒匹配一次；实际触发频率受 Agent Server 调度轮询影响。
test_cron = await client.crons.create_for_thread(
    assistant_id=assistant_id,
    thread_id=thread_id,
    schedule="* * * * * *", 
    input={},
    timezone="Asia/Shanghai",
)

# 生产：每天北京时间 03:00、04:00、05:00 各运行一次。
# production_cron = await client.crons.create(
#     assistant_id="delete_old_trace",
#     schedule="0 3-5 * * *",
#     input={},
#     timezone="Asia/Shanghai",
# )